# The Pensieve Project — DFIR + OSINT Lab\nCore leve primeiro. GLiNER e Qwen são opcionais para não transformar o Colab em um concurso de download de modelos.\n

In [ ]:
%pip -q install git+https://github.com/Ridd1kulusC0d3r/ThePensieveProject.git\n!pensieve-timeline doctor\n

## Laboratório sintético\nOs dados abaixo são fictícios. O objetivo é aprender o pipeline sem expor dados pessoais reais.\n

In [ ]:
from pathlib import Path\nPath('lab.csv').write_text('Timestamp,Computer,Channel,EventID,User,ProcessId,Details\\n2026-01-01T10:00:00Z,LAB-01,Security,4688,alice,4242,"powershell.exe contacted 10.10.10.5 and https://example.org/payload"\\n2026-01-01T10:00:05Z,LAB-01,ChatExport,,alice,,"contact ops@example.org and handle @blue_team"\\n2026-01-01T10:01:00Z,LAB-01,Sysmon,1,alice,4242,"Process related to example.org"\\n', encoding='utf-8')\n!pensieve-timeline ingest lab.csv -o timeline.jsonl --sqlite case.db\n!pensieve-timeline entities timeline.jsonl -o entities.jsonl\n!pensieve-timeline graph entities.jsonl -o entity-graph.json\n!pensieve-timeline report timeline.jsonl --entities entities.jsonl -o report.html\n

In [ ]:
from IPython.display import HTML, display\ndisplay(HTML(Path('report.html').read_text(encoding='utf-8')))\n

## GLiNER opcional\nAtive somente quando quiser NER zero-shot local. A saída continua ancorada em `event_id`.\n

In [ ]:
USE_GLINER = False\nif USE_GLINER:\n    %pip -q install gliner\n    !pensieve-timeline entities timeline.jsonl --gliner --labels person organization 'phone number' email domain 'social media handle' -o entities-gliner.jsonl\n

## Qwen opcional\nO modelo recebe um Evidence Packet reduzido, não o `raw` completo do caso.\n

In [ ]:
USE_QWEN = False\nif USE_QWEN:\n    %pip -q install 'transformers>=4.51,<5' 'accelerate>=1,<2'\n    !pensieve-timeline llm-summary timeline.jsonl entities.jsonl --model Qwen/Qwen3-0.6B -o intelligence.json\n    print(Path('intelligence.json').read_text(encoding='utf-8'))\n

## Checkpoint investigativo\n1. O que foi observado literalmente?\n2. O que foi apenas correlacionado?\n3. Qual hipótese alternativa continua plausível?\n4. Que nova evidência reduziria a incerteza?\n